### Laden und Protokoll vorbereiten

In [14]:
import pandas as pd
from dateutil.easter import easter
from datetime import timedelta

hotel = pd.read_csv("../data/processed/hotellerie_monat.csv")
herkunft = pd.read_csv("../data/processed/herkunft_monat.csv")
wetter = pd.read_csv("../data/processed/wetter_monat.csv")
gem = pd.read_csv("../data/processed/gemeinden.csv")

log = []
def schritt(name, vorher, nachher, grund):
    log.append({"schritt": name, "vorher": vorher, "nachher": nachher,
                "differenz": nachher - vorher, "grund": grund})

for name, df in [("hotel", hotel), ("herkunft", herkunft), ("wetter", wetter), ("gem", gem)]:
    print(name, df.shape)
    print(df.dtypes.to_dict())

hotel (29459, 9)
{'bfs_nr': dtype('int64'), 'jahr': dtype('int64'), 'monat': dtype('int64'), 'betten': dtype('float64'), 'bettenauslastung': dtype('float64'), 'logiernaechte': dtype('float64'), 'zimmer': dtype('float64'), 'zimmerauslastung': dtype('float64'), 'zimmernaechte': dtype('float64')}
herkunft (29378, 7)
{'bfs_nr': dtype('int64'), 'jahr': dtype('int64'), 'monat': dtype('int64'), 'ln_total': dtype('float64'), 'ln_schweiz': dtype('float64'), 'ln_ausland': dtype('float64'), 'anteil_ausland': dtype('float64')}
wetter (7200, 7)
{'bfs_nr': dtype('int64'), 'jahr': dtype('int64'), 'monat': dtype('int64'), 'temp_mittel': dtype('float64'), 'schneefall_cm': dtype('float64'), 'niederschlag_mm': dtype('float64'), 'schneetage': dtype('int64')}
gem (186, 12)
{'bfs_nr': dtype('int64'), 'name': <StringDtype(storage='python', na_value=nan)>, 'kanton': <StringDtype(storage='python', na_value=nan)>, 'lat': dtype('float64'), 'lon': dtype('float64'), 'hoehe_m': dtype('float64'), 'hoehenklasse': <St

### Plausibilität überprüfen

In [15]:
for name, df in [("hotel", hotel), ("herkunft", herkunft), ("wetter", wetter)]:
    print(name, "Duplikate:", df.duplicated(["bfs_nr", "jahr", "monat"]).sum())

print("Auslastung ausserhalb 0-100:", ((hotel["zimmerauslastung"] < 0) | (hotel["zimmerauslastung"] > 100)).sum())
print("Schneefall < 0:", (wetter["schneefall_cm"] < 0).sum())
w_winter = wetter[wetter["monat"].isin([12, 1, 2, 3])]
print("Wintertemperatur:", w_winter["temp_mittel"].min().round(1), "bis", w_winter["temp_mittel"].max().round(1))

hotel Duplikate: 0
herkunft Duplikate: 0
wetter Duplikate: 0
Auslastung ausserhalb 0-100: 0
Schneefall < 0: 0
Wintertemperatur: -11.3 bis 8.4


### Auswahl und Wintermonate / Ausnahmen

In [16]:
auswahl = gem.loc[gem["ausgewaehlt"], "bfs_nr"]

n0 = len(hotel)
p = hotel[hotel["bfs_nr"].isin(auswahl)]
schritt("Hotellerie: Auswahl Wintersportorte", n0, len(p), "186 auf 45 Gemeinden (Winteranteil >= 1/3)")

n1 = len(p)
p = p[p["monat"].isin([12, 1, 2, 3])].copy()
schritt("Hotellerie: nur Wintermonate", n1, len(p), "Dezember bis März")
print(len(p))

2375


In [17]:
p["winter"] = p["jahr"].where(p["monat"] == 12, p["jahr"] - 1).astype(int)
p["winter_label"] = p["winter"].astype(str) + "/" + (p["winter"] + 1).astype(str).str[-2:]

# Corona: Winter 2020/21 komplett und März 2020 (Winter 2019/20)
p["covid"] = ((p["winter"] == 2020) | ((p["winter"] == 2019) & (p["monat"] == 3))).astype(int)

# Ostern: Monat des Karfreitags (Ostern minus 2 Tage)
p["ostern_im_monat"] = p.apply(lambda r: int((easter(int(r.jahr)) - timedelta(days=2)).month == r.monat), axis=1)

print(p.groupby("winter_label").size())

winter_label
2012/13    120
2013/14    160
2014/15    160
2015/16    163
2016/17    170
2017/18    172
2018/19    175
2019/20    176
2020/21    179
2021/22    180
2022/23    180
2023/24    180
2024/25    180
2025/26    180
dtype: int64


### Unvollständige Winter entfernen

In [18]:
n2 = len(p)
p = p[p["winter"] >= 2013].copy()
schritt("Unvollständiger Winter 2012/13 entfernt", n2, len(p), "Dezember 2012 fehlt in den BFS-Daten")
print(pd.DataFrame(log))

                                   schritt  vorher  nachher  differenz  \
0      Hotellerie: Auswahl Wintersportorte   29459     7080     -22379   
1             Hotellerie: nur Wintermonate    7080     2375      -4705   
2  Unvollständiger Winter 2012/13 entfernt    2375     2255       -120   

                                        grund  
0  186 auf 45 Gemeinden (Winteranteil >= 1/3)  
1                           Dezember bis März  
2        Dezember 2012 fehlt in den BFS-Daten  


## Teil 2: Integration der Quellen

In [19]:
schl = ["bfs_nr", "jahr", "monat"]

n0 = len(p)
panel = p.merge(herkunft, on=schl, how="left", validate="1:1", indicator="m_herkunft")
ohne = (panel["m_herkunft"] == "left_only").sum()
schritt("+ Herkunft (Tabelle 101)", n0, len(panel),
        f"left join auf bfs_nr/jahr/monat; {ohne} Zeilen ohne Herkunftsdaten (BFS-Datenschutz)")
print(panel["m_herkunft"].value_counts())

m_herkunft
both          2255
left_only        0
right_only       0
Name: count, dtype: int64


In [20]:
n1 = len(panel)
panel = panel.merge(wetter, on=schl, how="left", validate="1:1", indicator="m_wetter")
ohne_w = (panel["m_wetter"] == "left_only").sum()
schritt("+ Wetter (Open-Meteo)", n1, len(panel),
        f"left join auf bfs_nr/jahr/monat; {ohne_w} Zeilen ohne Wetterdaten")
print(panel["m_wetter"].value_counts())      # left_only muss 0 sein

m_wetter
both          2255
left_only        0
right_only       0
Name: count, dtype: int64


In [21]:
n2 = len(panel)
stamm = gem[["bfs_nr", "name", "kanton", "lat", "lon", "hoehe_m",
             "hoehenklasse", "hoehenklasse_1000", "winteranteil", "korrigiert"]]
panel = panel.merge(stamm, on="bfs_nr", how="left", validate="m:1")
schritt("+ Gemeinde-Stammdaten", n2, len(panel), "left join auf bfs_nr (m:1); Höhe des Referenzorts")
print(panel["hoehenklasse"].isna().sum(), "Zeilen ohne Höhenklasse")     # muss 0 sein

0 Zeilen ohne Höhenklasse


In [22]:
n3 = len(panel)
fehl = panel[["zimmerauslastung", "schneefall_cm"]].isna().any(axis=1)
print(fehl.sum(), "Zeilen ohne Zielgrösse oder Wetter")
panel = panel[~fehl].copy()
schritt("Fehlende Zielgrösse/Wetter entfernt", n3, len(panel), "vertrauliche Werte ('...') im BFS")

0 Zeilen ohne Zielgrösse oder Wetter


In [23]:
diff = (panel["logiernaechte"] - panel["ln_total"]).abs()
print("Abweichungen logiernaechte (Tab. 201) vs ln_total (Tab. 101):", (diff > 0).sum())

Abweichungen logiernaechte (Tab. 201) vs ln_total (Tab. 101): 3


In [24]:
assert not panel.duplicated(schl).any()
panel = panel.drop(columns=["m_herkunft", "m_wetter"])
panel.to_csv("../data/processed/panel_winter.csv", index=False)
pd.DataFrame(log).to_markdown("../docs/integration_log.md", index=False)
print(panel.shape)
print(pd.DataFrame(log).to_string(index=False))

(2255, 30)
                                schritt  vorher  nachher  differenz                                                                           grund
    Hotellerie: Auswahl Wintersportorte   29459     7080     -22379                                      186 auf 45 Gemeinden (Winteranteil >= 1/3)
           Hotellerie: nur Wintermonate    7080     2375      -4705                                                               Dezember bis März
Unvollständiger Winter 2012/13 entfernt    2375     2255       -120                                            Dezember 2012 fehlt in den BFS-Daten
               + Herkunft (Tabelle 101)    2255     2255          0 left join auf bfs_nr/jahr/monat; 0 Zeilen ohne Herkunftsdaten (BFS-Datenschutz)
                  + Wetter (Open-Meteo)    2255     2255          0                      left join auf bfs_nr/jahr/monat; 0 Zeilen ohne Wetterdaten
                  + Gemeinde-Stammdaten    2255     2255          0                               lef

In [25]:
erw = 45 * 4 * 13
print(erw - len(panel), "fehlende Gemeinde-Wintermonate")
n_je_gem = panel.groupby("name").size()
print(n_je_gem[n_je_gem < 52].sort_values())          # 4 Monate x 13 Winter = 52
print(panel.groupby("winter_label").size())           # Winter 2025/26 vollständig?

85 fehlende Gemeinde-Wintermonate
name
Val de Bagnes    23
Rheinwald        31
Crans-Montana    39
Goms             39
Surses           43
dtype: int64
winter_label
2013/14    160
2014/15    160
2015/16    163
2016/17    170
2017/18    172
2018/19    175
2019/20    176
2020/21    179
2021/22    180
2022/23    180
2023/24    180
2024/25    180
2025/26    180
dtype: int64


In [26]:
panel["diff_ln"] = panel["logiernaechte"] - panel["ln_total"]
panel.loc[panel["diff_ln"].abs() > 0, ["name", "jahr", "monat", "logiernaechte", "ln_total", "diff_ln"]]

,name,jahr,monat,logiernaechte,ln_total,diff_ln
1011,Davos,2014,1,101743.0,102432.0,-689.0
1012,Davos,2014,12,114412.0,114824.0,-412.0
1013,Davos,2014,2,100776.0,101741.0,-965.0


### Dokumentation der Integration

**Schlüssel:** `bfs_nr` (BFS-Gemeindenummer), `jahr`, `monat`. Die Gemeindestammdaten werden über `bfs_nr` verknüpft.

**Joins:** Hotellerie (Tabelle 201) ist die Basis. Herkunft (Tabelle 101), Wetter (Open-Meteo) und Stammdaten werden per Left Join ergänzt. `validate` stellt sicher, dass keine Schlüssel doppelt vorkommen.

**Gelöste Inkonsistenzen:**
- Bei den 45 ausgewählten Gemeinden fehlen keine vertraulichen Werte (`...`) in Zimmerauslastung oder Wetter, es mussten keine Zeilen entfernt werden.
- 85 von 2'340 möglichen Gemeinde-Wintermonaten fehlen in den BFS-Daten (kein geöffneter Betrieb in diesem Monat, siehe Kontrolle).
- 3 Zeilen weichen in den Logiernächten zwischen Tabelle 201 und 101 ab (Differenz: ...).

**Verlorene und gewonnene Datensätze:** siehe Tabelle in `docs/integration_log.md`.

In [27]:
import pandas as pd
for f in ["gemeinden", "hotellerie_monat", "herkunft_monat", "wetter_monat", "panel_winter"]:
    d = pd.read_csv(f"../data/processed/{f}.csv")
    print(f"\n{f}: {d.shape}")
    print(d.dtypes.to_string())


gemeinden: (186, 12)
bfs_nr                 int64
name                     str
kanton                   str
lat                  float64
lon                  float64
hoehe_m              float64
hoehenklasse             str
hoehenklasse_1000        str
winteranteil         float64
ausgewaehlt             bool
ort_gefunden             str
korrigiert              bool

hotellerie_monat: (29459, 9)
bfs_nr                int64
jahr                  int64
monat                 int64
betten              float64
bettenauslastung    float64
logiernaechte       float64
zimmer              float64
zimmerauslastung    float64
zimmernaechte       float64

herkunft_monat: (29378, 7)
bfs_nr              int64
jahr                int64
monat               int64
ln_total          float64
ln_schweiz        float64
ln_ausland        float64
anteil_ausland    float64

wetter_monat: (7200, 7)
bfs_nr               int64
jahr                 int64
monat                int64
temp_mittel        float64
schne

In [28]:
import sqlite3, pathlib
import pandas as pd

schema = pathlib.Path("../sql/schema.sql").read_text()
con = sqlite3.connect("../data/processed/schnee.db")
con.executescript(schema)
print(pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name", con))
con.close()

             name   type
0        gemeinde  table
1        herkunft  table
2      hotellerie  table
3          wetter  table
4            zeit  table
5  v_panel_winter   view


In [35]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env")
engine = create_engine(
    f"postgresql+psycopg://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)

with engine.begin() as conn:
    for stmt in schema.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
    print(pd.read_sql(text("SELECT table_name, table_type FROM information_schema.tables WHERE table_schema = 'public' ORDER BY 2, 1"), conn))

       table_name  table_type
0        gemeinde  BASE TABLE
1        herkunft  BASE TABLE
2      hotellerie  BASE TABLE
3          wetter  BASE TABLE
4            zeit  BASE TABLE
5  v_panel_winter        VIEW
